# Distributed SPMD Cell Execution Demo

This notebook demonstrates NCCL communication across 4 GPUs using the distributed JupyterLab kernel.

## Setup

**Terminal 1:** Start JupyterLab
```bash
jupyter lab --ip=0.0.0.0
```

**Terminal 2:** Launch torchrun (copy session-config path from Terminal 1 log)
```bash
PYTHONPATH=/path/to/jupyterlab torchrun --nnodes 1 --nproc_per_node 4 --standalone \
    -m jupyterlab_distributed.launcher --session-config <path>
```

Then select **Distributed Python** kernel for this notebook.

## 1. Check Workers

In [ ]:
%distributed status

## 2. Init torch.distributed with NCCL

In [ ]:
import os, torch, torch.distributed as dist

local_rank = int(os.environ.get('LOCAL_RANK', '0'))
torch.cuda.set_device(local_rank)

if not dist.is_initialized():
    dist.init_process_group(backend='nccl')

rank = dist.get_rank()
world_size = dist.get_world_size()
gpu_name = torch.cuda.get_device_name(local_rank)
print(f"rank={rank}, world_size={world_size}, device=cuda:{local_rank}, gpu={gpu_name}")

## 3. NCCL AllReduce

Each rank creates a tensor with value `rank + 1`. After allreduce (SUM), all ranks should have `1+2+3+4 = 10`.

In [ ]:
t = torch.ones(4, device=f'cuda:{local_rank}') * (rank + 1)
print(f"rank={rank}: before allreduce: {t.tolist()}")

dist.all_reduce(t, op=dist.ReduceOp.SUM)

expected = float(sum(range(1, world_size + 1)))
print(f"rank={rank}: after allreduce: {t.tolist()} (expected={expected})")
assert t[0].item() == expected
print(f"rank={rank}: PASSED!")

## 4. NCCL AllGather

Each rank sends `[rank*10+1, rank*10+2]`. All ranks should receive the full gathered tensor.

In [ ]:
tensor = torch.tensor([rank * 10 + 1, rank * 10 + 2], device=f'cuda:{local_rank}', dtype=torch.float32)
gathered = [torch.zeros(2, device=f'cuda:{local_rank}') for _ in range(world_size)]
dist.all_gather(gathered, tensor)

result = [g.tolist() for g in gathered]
print(f"rank={rank}: input={tensor.tolist()}, allgather={result}")
print(f"rank={rank}: PASSED!")

## 5. Rank-0 Only (checkpoint example)

Use `%%rank0` to run a cell only on rank-0 — useful for saving checkpoints, logging, etc.

In [ ]:
%%rank0
print(f"Only rank-0 sees this!")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**2:.1f} MB")

## 6. Simple DDP Training Step

In [ ]:
from torch.nn.parallel import DistributedDataParallel as DDP

# Simple model
model = torch.nn.Linear(128, 64).to(f'cuda:{local_rank}')
model = DDP(model, device_ids=[local_rank])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Forward + backward
x = torch.randn(32, 128, device=f'cuda:{local_rank}')
loss = model(x).sum()
loss.backward()
optimizer.step()
optimizer.zero_grad()

print(f"rank={rank}: loss={loss.item():.4f}, params={sum(p.numel() for p in model.parameters())}")